# rsna-train-ttr-f34: TTR mix50 folds 3 + 4 (seed 1043) on Kaggle T4 x2

Same trainer, data, labels and pretrained bytes as the V100 box runs (J1 fold 0, folds 1-2), one fold per T4, in parallel.

1. locate inputs (code dataset `eeshsaxena/rsna-knee-fleet-train`, prep kernel outputs `rsna-fleet-prep-s0/-s1`), verify every file by sha256
2. environment report (timm < 1.0: bundled wheel)
3. extract the 12 store tars to `/tmp/fleet_store`, prove it equals the box `~/fleet_store` (names, 24 sampled files)
4. parity: contract pixel/tensor sha, select_sha, jpeg_fp, model_sha, decode probe, state_dict keys, pretrained weights, fold counts
5. timing probe: both trainers concurrently, steady s/it, CPU/GPU utilisation
6. `E = min(24, floor(budget / epoch time))`, launch, watchdog (hard wall 11.4 h), receipt
7. summary

Outputs: `f3_s1043_ttr_mix50/`, `f4_s1043_ttr_mix50/` (+ `.log`), `run_receipt.json`, `probe/`.


In [ ]:
# [lib] rsna-train-ttr-f34: helpers only (no GPU, no network, no side effects beyond defining things).
# Trains TTR mix50 folds 3 and 4 (seed 1043) with the box recipe, one fold per T4, in parallel.
# KT_* environment variables exist ONLY for the box dry test; a pushed Kaggle kernel receives none of them.
import os, sys, re, ast, json, math, time, glob, shutil, signal, hashlib, tarfile, datetime, subprocess
from concurrent.futures import ThreadPoolExecutor

T_NB0 = time.time()
KT_ROOT = os.environ.get('KT_ROOT', '')            # '' on Kaggle; the dry test maps /kaggle/... and /tmp/... under it


def P(p):
    return KT_ROOT + p


INPUT, WORK = P('/kaggle/input'), P('/kaggle/working')
STORE, CODE, HF_HOME, PROBE_DIR = P('/tmp/fleet_store'), P('/tmp/kt_code'), P('/tmp/kt_hf'), P('/tmp/kt_probe')
LBL = P('/tmp/kt_labels')                          # verified copies of the 4 label/fold csvs (+ box_facts.json)


def _env(k, d, f=float):
    return f(os.environ.get(k, d))


CFG = dict(
    FOLDS=[3, 4],
    GPU_IDS=os.environ.get('KT_GPU_IDS', '0,1').split(','),   # fold 3 -> GPU0, fold 4 -> GPU1
    SEED=1043, EPOCHS_MAX=24, EPOCHS_WARN=16,
    BUDGET_H=_env('KT_BUDGET_H', 10.3),     # training is planned to END by this many hours after kernel start
    WALL_H=_env('KT_WALL_H', 11.4),         # hard wall: trainers terminated (SIGTERM, then SIGKILL), outputs kept
    EVAL_FRAC=_env('KT_EVAL_FRAC', 0.21),   # eval time / train time per epoch; box J1 log: TRAIN-ONLY 10.1-10.4 min,
                                            # epoch incl. raw+ema eval (val 867 + gold74) 12.3-12.6 min -> 0.21
    STARTUP_S=_env('KT_STARTUP_S', 120),    # trainer start: labels, contract probe of 8 npz, model build, workers
    WORKERS=_env('KT_WORKERS', 3, int),     # DataLoader workers PER trainer (Kaggle T4x2 = 4 vCPU for both)
    PROBE_ITERS=_env('KT_PROBE_ITERS', 400, int),      # trainer prints s/it every 200 it; 400 = window 200 -> 400
    PROBE_TIMEOUT_S=_env('KT_PROBE_TIMEOUT_S', 1500),
    PROBE_POLL_S=_env('KT_PROBE_POLL_S', 5),
    MIN_FREE_GB=_env('KT_MIN_FREE_GB', 20),
    KILL_GRACE_S=_env('KT_KILL_GRACE_S', 60),
    POLL_S=_env('KT_POLL_S', 30),
    STATUS_EVERY_S=_env('KT_STATUS_EVERY_S', 900),
    OUT_LIMIT_GB=_env('KT_OUT_LIMIT_GB', 19),
    SKIP_GPU_CHECK=os.environ.get('KT_SKIP_GPU_CHECK', '') == '1',
    TRAINER=os.environ.get('KT_TRAINER_PY', ''),        # '' = the real fleet_train.py (dry test: a dummy)
)
FOLDS = CFG['FOLDS']
RUN = {F: f'f{F}_s{CFG["SEED"]}_ttr_mix50' for F in FOLDS}
CODE_FILES = ['fleet_train.py', 'fleet_common.py', 'fleet_model.py', 'fleet_model_ttr.py', 'fleet_model_f2.py',
              'fleet_aug.py']
LABEL_FILES = ['labels_mix50_logit.csv', 'folds5.csv', 'gold74_eval_set.csv', 'llm_labels_v4_blend.csv']
TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA',
           'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
TAR_RE = re.compile(r'fleet_store_s[01]_p\d\d\.tar')
NPZ_RE = re.compile(r'[0-9][0-9.]*\.npz')


def _proc_start(pid):
    up = float(open('/proc/uptime').read().split()[0])
    st = int(open(f'/proc/{pid}/stat').read().rsplit(')', 1)[1].split()[19]) / os.sysconf('SC_CLK_TCK')
    return time.time() - max(0.0, up - st)


def kernel_t0():
    """Kaggle's 12 h clock starts with the session: the Jupyter kernel process start (the K2 cell's proven method),
    or the container's pid 1 when that started up to 30 min earlier (more conservative), never later than now."""
    if os.environ.get('KT_T0'):
        return float(os.environ['KT_T0']), 'env KT_T0'
    try:
        t_self = min(_proc_start('self'), T_NB0)
    except Exception:
        return T_NB0, 'notebook start'
    try:
        t1 = _proc_start(1)
        if 0 <= t_self - t1 <= 1800:
            return t1, 'container pid 1 start'
    except Exception:
        pass
    return t_self, 'kernel process start'


T0, T0_SRC = kernel_t0()
IST = datetime.timezone(datetime.timedelta(hours=5, minutes=30))


def ist(t=None):
    return datetime.datetime.fromtimestamp(time.time() if t is None else t, IST).strftime('%m-%d %H:%M IST')


def el():
    return time.time() - T0


def log(*a):
    print(f'[kt {el() / 3600:5.2f}h {ist()}]', *a, flush=True)


REC = dict(kernel='eeshsaxena/rsna-train-ttr-f34', status='starting', t0=T0, t0_src=T0_SRC, t0_ist=ist(T0),
           cfg=CFG, runs=RUN, warnings=[], stages={})


def save_receipt():
    os.makedirs(WORK, exist_ok=True)
    REC['updated_ist'] = ist()
    REC['elapsed_h'] = round(el() / 3600, 4)
    tmp = f'{WORK}/run_receipt.json.tmp'
    with open(tmp, 'w') as f:
        json.dump(REC, f, indent=1, default=str)
    os.replace(tmp, f'{WORK}/run_receipt.json')


def warn(m):
    REC['warnings'].append(f'{ist()} {m}')
    log('WARNING', m)
    save_receipt()


def stage(name, **kw):
    REC['stages'][name] = dict(kw, at_h=round(el() / 3600, 4))
    save_receipt()


def abort(m):
    REC['status'] = f'ABORTED: {m}'
    log('ABORT', m)
    save_receipt()
    raise RuntimeError(m)


def sha256_file(p, bs=1 << 22):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(bs), b''):
            h.update(b)
    return h.hexdigest()


def read_text(p):
    try:
        with open(p, errors='replace') as f:
            return f.read()
    except FileNotFoundError:
        return ''


def tail(p, n=25):
    return '\n'.join(read_text(p).splitlines()[-n:])


# ---------------------------------------------------------------- inputs
def walk_find(root, pred, maxdepth=6):
    out, base = [], os.path.abspath(root).rstrip(os.sep).count(os.sep)
    for d, dirs, files in os.walk(root, followlinks=True):
        if os.path.abspath(d).count(os.sep) - base >= maxdepth:
            dirs[:] = []
        for f in files:
            if pred(d, f):
                out.append(os.path.join(d, f))
    return sorted(out)


def locate_inputs():
    """-> (dataset dir, {tar name: path}, {shard: manifest path}). Handles /kaggle/input/<slug>/ and
    /kaggle/input/{datasets,notebooks}/<owner>/<slug>/ (and anything else up to depth 6)."""
    ds = sorted({os.path.realpath(os.path.dirname(p)) for p in
                 walk_find(INPUT, lambda d, f: f == 'kt_manifest.json' and os.path.exists(f'{d}/fleet_train.py'))})
    if len(ds) > 1 and len({open(f'{d}/kt_manifest.json', 'rb').read() for d in ds}) == 1:
        warn(f'code dataset visible at {len(ds)} paths with identical kt_manifest.json (legacy + new mount layout): '
             f'using {ds[0]}')       # every file is still sha256-checked against that manifest below
        ds = ds[:1]
    if len(ds) != 1:
        abort(f'code dataset eeshsaxena/rsna-knee-fleet-train: found {ds} under {INPUT} (want exactly 1)')
    tars, mans = {}, {}
    for p in walk_find(INPUT, lambda d, f: bool(TAR_RE.fullmatch(f))):
        n = os.path.basename(p)
        if n in tars and os.path.realpath(tars[n]) != os.path.realpath(p):
            warn(f'tar {n} found twice: {tars[n]} and {p}; using the first')
        tars.setdefault(n, p)
    for p in walk_find(INPUT, lambda d, f: re.fullmatch(r'prep_manifest_[01]\.json', f) is not None):
        mans.setdefault(int(p[-6]), p)
    return ds[0], tars, mans


# ---------------------------------------------------------------- store extraction
class _HashReader:
    def __init__(self, f, h):
        self.f, self.h, self.n = f, h, 0

    def read(self, n=-1):
        b = self.f.read(n)
        self.h.update(b)
        self.n += len(b)
        return b


def extract_tar(name, path, dst):
    """Stream one prep tar into dst, hashing the WHOLE file on the way (one read). Members must be flat
    '<StudyInstanceUID>.npz' names. -> (name, sha256, n_members, bytes)."""
    h = hashlib.sha256()
    n = 0
    with open(path, 'rb') as f:
        r = _HashReader(f, h)
        with tarfile.open(fileobj=r, mode='r|') as t:
            for m in t:
                if not m.isfile():
                    continue
                if not NPZ_RE.fullmatch(m.name):
                    raise ValueError(f'{name}: unexpected member {m.name!r}')
                src = t.extractfile(m)
                tmp = f'{dst}/{m.name}.part'
                with open(tmp, 'wb') as o:
                    shutil.copyfileobj(src, o, 1 << 22)
                os.replace(tmp, f'{dst}/{m.name}')
                n += 1
        while r.read(1 << 22):           # tar end padding: hash the whole file
            pass
    return name, h.hexdigest(), n, r.n


# ---------------------------------------------------------------- fold arithmetic (same as fleet_train.py)
def expected_counts(ds_dir, store, F):
    import pandas as pd
    lab = pd.read_csv(f'{ds_dir}/labels_mix50_logit.csv', dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')
    folds = pd.read_csv(f'{ds_dir}/folds5.csv', dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')['fold']
    excl = set(pd.read_csv(f'{ds_dir}/gold74_eval_set.csv', dtype={'StudyInstanceUID': str}).StudyInstanceUID)
    held = set(folds.index[folds < 0]) | excl
    have = {s for s in lab.index if os.path.exists(f'{store}/{s}.npz')}
    ids = [s for s in lab.index if s in folds.index and s in have]
    tr = [s for s in ids if s not in held and folds[s] != F]
    va = [s for s in ids if s not in held and folds[s] == F]
    return dict(labels=len(lab), store_has=len(have), held=len(held), train=len(tr), val=len(va),
                iters_per_epoch=len(tr) // 2)


# ---------------------------------------------------------------- trainer log parsing
RE_IT = re.compile(r'^ep (\d+) it (\d+)/(\d+) loss \S+ ([0-9.]+)s/it', re.M)
RE_LAB = re.compile(r'^labels (\d+) \| store has (\d+) \| held-out \(fold<0 or --exclude\) (\d+) \| train (\d+) \| '
                    r'val (\d+) \| all_data (\w+) \| device (\w+)', re.M)
RE_CON = re.compile(r'^contract (\{.*\}) \| select_sha (\w+) \| jpeg_fp (\w+)\s*$', re.M)
RE_BJ = re.compile(r'^BJ opt-in: .*\| model_sha (\w+)\s*$', re.M)
RE_EV = re.compile(r'^ep (\d+) (raw|ema) \|.*\| val AUC vs \S+ (\S+) \((\d+) labels\) \| vs train labels (\S+) \| '
                   r'epoch ([0-9.]+) min\s*$', re.M)
RE_TO = re.compile(r'^ep (\d+) TRAIN-ONLY ([0-9.]+) min', re.M)


def parse_its(text, ep=0):
    """-> [(it, n_iters, s_per_it_cumulative)] for one epoch, in log order."""
    return [(int(m.group(2)), int(m.group(3)), float(m.group(4))) for m in RE_IT.finditer(text) if int(m.group(1)) == ep]


def sit_window(its):
    """Steady s/it from the LAST two progress lines. The trainer prints (now - epoch_start) / (it + 1), so the
    cumulative time at line `it` is (it + 1) * s; the window drops the first-iteration (cudnn benchmark) cost."""
    if len(its) < 2:
        return None
    (n1, _, s1), (n2, _, s2) = its[-2], its[-1]
    return ((n2 + 1) * s2 - (n1 + 1) * s1) / (n2 - n1)


def parse_header(text):
    """The trainer's start lines -> dict (None for lines not printed yet)."""
    h = dict(labels=None, contract=None, select_sha=None, jpeg_fp=None, model_sha=None)
    m = RE_LAB.search(text)
    if m:
        h['labels'] = dict(labels=int(m.group(1)), store_has=int(m.group(2)), held=int(m.group(3)),
                           train=int(m.group(4)), val=int(m.group(5)), all_data=m.group(6), device=m.group(7))
    m = RE_CON.search(text)
    if m:
        h['contract'] = ast.literal_eval(m.group(1))
        h['select_sha'], h['jpeg_fp'] = m.group(2), m.group(3)
    m = RE_BJ.search(text)
    if m:
        h['model_sha'] = m.group(1)
    return h


def header_problems(h, exp_counts, box):
    """Compare a parsed trainer header with the box contract and the expected fold counts. -> list of problems
    ([] = OK). Lines not printed yet are reported as missing."""
    bad = []
    a = box['ckpt_args']
    if h['contract'] is None:
        bad.append('contract line missing')
    else:
        for k in ('pixel_sha', 'tensor_sha'):
            if h['contract'].get(k) != a['contract'][k]:
                bad.append(f'{k} {h["contract"].get(k)} != box {a["contract"][k]}')
        if json.loads(json.dumps(h['contract'])) != a['contract']:
            bad.append(f'contract {h["contract"]} != box {a["contract"]}')
        if h['select_sha'] != a['select_sha']:
            bad.append(f'select_sha {h["select_sha"]} != box {a["select_sha"]}')
        if h['jpeg_fp'] != a['jpeg_fp']:
            bad.append(f'jpeg_fp {h["jpeg_fp"]} != box {a["jpeg_fp"]}')
    if h['labels'] is None:
        bad.append('labels line missing')
    else:
        for k in ('labels', 'store_has', 'held', 'train', 'val'):
            if h['labels'][k] != exp_counts[k]:
                bad.append(f'{k} {h["labels"][k]} != expected {exp_counts[k]}')
        if h['labels']['all_data'] != 'False':
            bad.append('all_data is not False')
    if h['model_sha'] is None:
        bad.append('BJ opt-in / model_sha line missing')
    elif h['model_sha'] != a['model_sha']:
        bad.append(f'model_sha {h["model_sha"]} != box {a["model_sha"]}')
    return bad


def progress(text):
    """-> dict(evals=[(ep, tag, val, epoch_min)], last_ep_done, train_only_min=[...], best={tag: (val, ep)})."""
    ev = [(int(m.group(1)), m.group(2), m.group(3), float(m.group(6))) for m in RE_EV.finditer(text)]
    best = {}
    for ep, tag, v, _ in ev:
        try:
            fv = float(v)
        except ValueError:
            continue
        if math.isfinite(fv) and (tag not in best or fv > best[tag][0]):
            best[tag] = (fv, ep)
    done = [ep for ep, tag, _, _ in ev if tag == 'ema']
    return dict(evals=ev, last_ep_done=max(done) if done else -1, best=best,
                train_only_min=[float(m.group(2)) for m in RE_TO.finditer(text)])


def closing_line(text):
    """fleet_train.py ends with json.dumps(best) = {"raw": .., "ema": ..}. None when the run did not finish."""
    lines = [l for l in text.splitlines() if l.strip()]
    if not lines:
        return None
    try:
        j = json.loads(lines[-1])
    except Exception:
        return None
    return j if isinstance(j, dict) and set(j) == {'raw', 'ema'} else None


# ---------------------------------------------------------------- epochs
def choose_epochs(s_it, iters, elapsed_s, budget_h=None, eval_frac=None, startup_s=None, emax=None):
    """E = min(EPOCHS_MAX, floor(time left in the budget / per-epoch time incl. eval)), at least 1."""
    budget_h = CFG['BUDGET_H'] if budget_h is None else budget_h
    eval_frac = CFG['EVAL_FRAC'] if eval_frac is None else eval_frac
    startup_s = CFG['STARTUP_S'] if startup_s is None else startup_s
    emax = CFG['EPOCHS_MAX'] if emax is None else emax
    epoch_s = iters * s_it * (1.0 + eval_frac)
    avail_s = budget_h * 3600.0 - elapsed_s - startup_s
    E = int(max(1, min(emax, math.floor(avail_s / epoch_s)))) if epoch_s > 0 else emax
    return dict(E=E, epoch_s=epoch_s, avail_s=avail_s, finish_elapsed_s=elapsed_s + startup_s + E * epoch_s)


def save_epochs_arg(E):
    return f'{max(1, E - 1)}-{E}'          # per-epoch ckpts for the LAST 2 epochs only (2 x 2 x 113 MB per fold)


# ---------------------------------------------------------------- processes
HF_OFFLINE = ['1']                  # trainers load the bundled pretrained bytes from HF_HOME; '0' only on fallback


def recipe_args(F, E, save_ep, out, lbl_dir, workers):
    L = lbl_dir
    return ['--model', 'ttr', '--aug', 'heavy', '--train_slices', '16', '--drop_path', '0.2', '--dropout', '0.3',
            '--ema', '0.9995', '--epochs', str(E), '--seed', str(CFG['SEED']), '--save_epochs', save_ep,
            '--workers', str(workers),
            '--store', STORE, '--labels', f'{L}/labels_mix50_logit.csv', '--folds', f'{L}/folds5.csv',
            '--exclude', f'{L}/gold74_eval_set.csv', '--eval_labels', f'{L}/llm_labels_v4_blend.csv',
            '--gold', f'{L}/gold74_eval_set.csv', '--fold', str(F), '--out', out]


def launch(F, gpu, E, save_ep, out, logp, lbl_dir, workers):
    os.makedirs(os.path.dirname(logp), exist_ok=True)
    cmd = [sys.executable, CFG['TRAINER'] or f'{CODE}/fleet_train.py'] + recipe_args(F, E, save_ep, out, lbl_dir, workers)
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED='1', OMP_NUM_THREADS='1',
               MKL_NUM_THREADS='1', HF_HOME=HF_HOME, HF_HUB_OFFLINE=HF_OFFLINE[0],
               PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
    lf = open(logp, 'w')
    p = subprocess.Popen(cmd, cwd=CODE, stdout=lf, stderr=subprocess.STDOUT, env=env, start_new_session=True)
    p.kt_log, p.kt_cmd = lf, cmd
    log(f'launched fold {F} on GPU {gpu}: pid {p.pid} | log {logp}')
    return p


def stop_group(p, grace):
    """SIGTERM the trainer's whole process group (main + DataLoader workers), SIGKILL after `grace` s.
    -> 'exited' | 'sigterm' | 'sigkill'."""
    how = 'exited'
    if p.poll() is None:
        how = 'sigterm'
        try:
            os.killpg(p.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            p.wait(grace)
        except subprocess.TimeoutExpired:
            how = 'sigkill'
            try:
                os.killpg(p.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            p.wait()
    try:
        os.killpg(p.pid, signal.SIGKILL)          # stray workers of a main that already exited
    except (ProcessLookupError, PermissionError):
        pass
    try:
        p.kt_log.close()
    except Exception:
        pass
    return how


def wait_quiet(dirs, quiet_s=15.0, max_wait_s=120.0, poll_s=2.0):
    """Before the hard-wall SIGTERM. fleet_train.save() first DELETES the previous best ckpt of the tag, then torch.save()s
    the new one straight to its final name (no tmp + rename), then writes the OOF csv: a kill inside that window leaves a
    truncated best ckpt (or a ckpt/OOF pair from different epochs). Wait (bounded) until nothing under `dirs` (files AND
    directories: a delete changes the dir mtime) changed for `quiet_s` s. Saves only follow an eval line and take seconds.
    -> seconds waited."""
    t = time.time()
    while True:
        newest = 0.0
        for d in dirs:
            for root, _, fs in os.walk(d):
                for p in [root] + [os.path.join(root, f) for f in fs]:
                    try:
                        newest = max(newest, os.path.getmtime(p))
                    except OSError:
                        pass
        if time.time() - newest >= quiet_s or time.time() - t >= max_wait_s:
            return time.time() - t
        time.sleep(poll_s)


def gpu_query():
    try:
        out = subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total,memory.used,utilization.gpu',
                              '--format=csv,noheader,nounits'], capture_output=True, text=True, timeout=60).stdout
    except Exception:
        return []
    keys = ['index', 'name', 'mem_total_mib', 'mem_used_mib', 'util_pct']
    return [dict(zip(keys, [x.strip() for x in l.split(',')])) for l in out.strip().splitlines() if l.count(',') == 4]


def wait_gpus_free(timeout=120, max_used_mib=1000):
    if CFG['SKIP_GPU_CHECK']:
        return True
    t = time.time()
    while time.time() - t < timeout:
        g = gpu_query()
        if g and all(float(x['mem_used_mib']) < max_used_mib for x in g):
            return True
        time.sleep(3)
    return False


def cpu_times():
    v = [int(x) for x in open('/proc/stat').readline().split()[1:]]
    return sum(v[:8]), v[3] + v[4]


def cpu_busy(a, b):
    return 1.0 - (b[1] - a[1]) / max(1, b[0] - a[0])


MODELCHECK_SRC = r'''
import os, sys, json
sys.path.insert(0, sys.argv[1])
import torch, timm
from fleet_model_ttr import KneeFleetNetTTR
box = json.load(open(sys.argv[2]))
m = KneeFleetNetTTR(ttr_size=224, backbone='convnext_tiny.fb_in22k_ft_in1k', dim=256, contrast_emb=True, n_views=4,
                    n_slices=24, n_contrast=4, pretrained=True, drop_path=0.2, dropout=0.3, grad_ckpt=True)
got = [[k, list(v.shape), str(v.dtype).replace('torch.', '')] for k, v in m.state_dict().items()]
exp = box['state_dict']
res = dict(timm=timm.__version__, torch=torch.__version__, n_keys=len(got), n_keys_box=len(exp), keys_match=got == exp)
if got != exp:
    gs, es = {g[0]: g for g in got}, {e[0]: e for e in exp}
    res['only_here'] = sorted(set(gs) - set(es))[:10]; res['only_box'] = sorted(set(es) - set(gs))[:10]
    res['shape_or_dtype_diff'] = [k for k in gs if k in es and gs[k] != es[k]][:10]
from safetensors.torch import load_file
pre = load_file(sys.argv[3]); cnn = m.cnn.state_dict()
miss = [k for k in cnn if k not in pre]; neq = [k for k in cnn if k in pre and not torch.equal(cnn[k].float(), pre[k].float())]
res.update(cnn_keys=len(cnn), n_missing_in_pretrained=len(miss), n_not_equal=len(neq), missing=miss[:5], not_equal=neq[:5],
           pretrained_equal=(not miss and not neq))
print('KTMODELCHECK ' + json.dumps(res), flush=True)
'''


def install_hf_cache(ds_dir, exp):
    """Put the box's own convnext_tiny.fb_in22k_ft_in1k bytes into an HF hub cache, so timm's pretrained=True
    resolves them offline (HF_HUB_OFFLINE=1) exactly like the box (~/.cache/huggingface/hub)."""
    hf = exp['hf']
    repo = f'{HF_HOME}/hub/models--{hf["repo"].replace("/", "--")}'
    os.makedirs(f'{repo}/refs', exist_ok=True)
    os.makedirs(f'{repo}/snapshots/{hf["revision"]}', exist_ok=True)
    with open(f'{repo}/refs/main', 'w') as f:
        f.write(hf['revision'])
    dst = f'{repo}/snapshots/{hf["revision"]}/model.safetensors'
    shutil.copyfile(f'{ds_dir}/{hf["file"]}', dst)
    assert sha256_file(dst) == hf['sha256'], 'pretrained bytes changed while copying'
    return dst


In [ ]:
# [step 1] locate the attached inputs, verify every dataset file against kt_manifest.json, copy the trainer code.
log(f'kernel t0 {ist(T0)} ({T0_SRC}); notebook start +{(T_NB0 - T0) / 60:.1f} min; root {KT_ROOT or "/"}')
save_receipt()
DS, TARS, MANS = locate_inputs()
EXP = json.load(open(f'{DS}/kt_expect.json'))
BOX = json.load(open(f'{DS}/box_facts.json'))
log(f'dataset {DS} | {len(TARS)} tar parts | manifests {sorted(MANS)}')
for n in sorted(TARS):
    log(f'  {n}  {os.path.getsize(TARS[n]) / 1e9:.3f} GB  {TARS[n]}')

_man = json.load(open(f'{DS}/kt_manifest.json'))
_soft = set(LABEL_FILES) | {EXP['timm_wheel']}        # csvs: checked below (CRLF-tolerant); wheel: optional fallback
_bad = [f for f, h in _man['files'].items()
        if f not in _soft and (not os.path.exists(f'{DS}/{f}') or sha256_file(f'{DS}/{f}') != h)]
if _bad:
    abort(f'dataset files missing or changed vs kt_manifest.json: {_bad}')
_w = EXP['timm_wheel']
WHEEL_OK = os.path.exists(f'{DS}/{_w}') and sha256_file(f'{DS}/{_w}') == _man['files'][_w]
if not WHEEL_OK:
    warn(f'{_w} missing or changed: the timm fallback would use PyPI (only needed if timm < 1.0 here)')
log(f'dataset files verified: {len(_man["files"]) - len(_soft)} sha256 match kt_manifest.json (+ csvs below)')

_want = set(EXP['parts'])
if set(TARS) != _want:
    abort(f'prep tar parts: missing {sorted(_want - set(TARS))}, unexpected {sorted(set(TARS) - _want)} '
          f'(attach kernel outputs eeshsaxena/rsna-fleet-prep-s0 and -s1)')
for s, mp in sorted(MANS.items()):
    m = json.load(open(mp))
    for n, h in m['parts'].items():
        if EXP['parts'].get(n) != h:
            abort(f'{mp}: part {n} sha {h} != expected {EXP["parts"].get(n)}')
if sorted(MANS) != [0, 1]:
    warn(f'prep manifests found for shards {sorted(MANS)} only (tar hashes are still checked against kt_expect.json)')

shutil.rmtree(CODE, ignore_errors=True)
os.makedirs(CODE)
for f in CODE_FILES:
    shutil.copy2(f'{DS}/{f}', f'{CODE}/{f}')
    if sha256_file(f'{CODE}/{f}') != BOX['code_sha256'][f]:
        abort(f'{f} differs from the box copy (~/bigjump_build/code/{f})')
with open(f'{CODE}/kt_modelcheck.py', 'w') as f:
    f.write(MODELCHECK_SRC)
shutil.rmtree(LBL, ignore_errors=True)
os.makedirs(LBL)
for f in LABEL_FILES:                       # the trainer reads these copies; bytes == the box ~/rsna_train files
    b = open(f'{DS}/{f}', 'rb').read()
    if hashlib.sha256(b).hexdigest() != EXP['labels_sha256'][f]:
        b = b.replace(bytes([13, 10]), bytes([10]))          # CRLF -> LF
        if hashlib.sha256(b).hexdigest() != EXP['labels_sha256'][f]:
            abort(f'{f} differs from the box ~/rsna_train/{f}')
        warn(f'{f}: CRLF line endings on Kaggle; LF-normalised copy == the box bytes, using it')
    with open(f'{LBL}/{f}', 'wb') as o:
        o.write(b)
shutil.copy2(f'{DS}/box_facts.json', f'{LBL}/box_facts.json')
REC['inputs'] = dict(dataset=DS, labels_dir=LBL, tars=TARS, manifests=MANS, code_sha256=BOX['code_sha256'],
                     labels_sha256=EXP['labels_sha256'], wheel_ok=WHEEL_OK)
stage('inputs', ok=True)
log('code == box ~/bigjump_build/code (6 files), labels == box ~/rsna_train (4 files)')


In [ ]:
# [step 2] environment: versions, GPUs, CPU, RAM, disk. timm < 1.0 or missing -> install the bundled timm 1.0.29
# wheel (no internet needed), pip from PyPI only as a second fallback (internet is on). Logged either way.
import importlib


def _vt(v):
    return tuple(int(x) for x in re.findall(r'\d+', v)[:3])


try:
    import timm
    _tv = timm.__version__
except Exception:
    _tv = None
if _tv is None or _vt(_tv) < (1, 0):
    _rc = 1
    if WHEEL_OK:                                        # stored as .whl.bin so Kaggle never unpacks it
        os.makedirs(P('/tmp/kt_whl'), exist_ok=True)
        _whl = P('/tmp/kt_whl/') + EXP['timm_wheel'][:-len('.bin')]
        shutil.copyfile(f'{DS}/{EXP["timm_wheel"]}', _whl)
        _rc = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '--quiet', _whl]).returncode
    if _rc != 0:
        _rc = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'timm==1.0.29']).returncode
    warn(f'timm was {_tv}: pip install timm 1.0.29 rc {_rc}')
    importlib.invalidate_caches()
    for _m in [m for m in list(sys.modules) if m == 'timm' or m.startswith('timm.')]:
        del sys.modules[_m]
    import timm
import numpy as np, pandas as pd, torch, cv2, sklearn
VERS = dict(python=sys.version.split()[0], torch=torch.__version__, torch_cuda=torch.version.cuda, timm=timm.__version__,
            numpy=np.__version__, cv2=cv2.__version__, sklearn=sklearn.__version__, pandas=pd.__version__)
if VERS['timm'] != EXP['box_versions']['timm']:
    log(f'note: timm {VERS["timm"]} here vs {EXP["box_versions"]["timm"]} on the box (the state_dict check below decides)')
GPUS = gpu_query()
_mem = {l.split(':')[0]: l.split(':')[1].strip() for l in open('/proc/meminfo') if ':' in l}
_free = {p: round(shutil.disk_usage(p).free / 1e9, 1) for p in (P('/tmp'), WORK, P('/dev/shm')) if os.path.exists(p)}
if _free.get(P('/dev/shm'), 99) < 2:      # DataLoader batches travel through /dev/shm; the probe exercises train batches
    warn(f'/dev/shm has only {_free[P("/dev/shm")]} GB free: the eval loaders (not exercised by the probe) may fail')
ENV = dict(versions=VERS, box_versions=EXP['box_versions'], gpus=GPUS, cpu_count=os.cpu_count(),
           mem_total=_mem.get('MemTotal'), mem_available=_mem.get('MemAvailable'), disk_free_gb=_free)
REC['env'] = ENV
log('versions', json.dumps(VERS))
log('box     ', json.dumps(EXP['box_versions']))
log(f'GPUs {[(g["index"], g["name"], g["mem_used_mib"] + "/" + g["mem_total_mib"] + " MiB") for g in GPUS]} | '
    f'cpus {os.cpu_count()} | RAM {_mem.get("MemTotal")} (avail {_mem.get("MemAvailable")}) | disk free GB {_free}')
if not CFG['SKIP_GPU_CHECK']:
    if len(GPUS) < len(FOLDS):
        abort(f'{len(GPUS)} GPU(s) visible, need {len(FOLDS)} (kernel-metadata machine_shape NvidiaTeslaT4 = T4 x2)')
    if any('T4' not in g['name'] for g in GPUS):
        warn(f'GPU is not a T4: {[g["name"] for g in GPUS]} (timing calibration still applies)')
    if any(float(g['mem_used_mib']) > 1000 for g in GPUS):
        abort(f'GPU memory already in use before training: {GPUS}')
stage('env', ok=True)


In [ ]:
# [step 3] extract the 12 prep tars to <scratch>/fleet_store (hash each tar while streaming it), then prove the store
# is the box's ~/fleet_store: same 4407 names, 24 sampled files byte-identical (sha256), every labelled study present.
# Scratch = the existing one of /tmp, /kaggle/tmp, /kaggle/temp with the most free space (never /kaggle/working:
# that is the 20 GB output area).
_free = {d: shutil.disk_usage(d).free / 1e9 for d in (P('/tmp'), P('/kaggle/tmp'), P('/kaggle/temp')) if os.path.isdir(d)}
_par = max(_free, key=_free.get)
STORE = f'{_par}/fleet_store'
_free_gb = _free[_par]
_need_gb = sum(os.path.getsize(p) for p in TARS.values()) / 1e9
log(f'disk free GB {({k: round(v, 1) for k, v in _free.items()})}: store -> {STORE} (needs ~{_need_gb:.1f} GB; '
    f'floor {CFG["MIN_FREE_GB"]} GB)')
if _free_gb < CFG['MIN_FREE_GB']:
    abort(f'only {_free_gb:.1f} GB free at {_par} (< {CFG["MIN_FREE_GB"]} GB; the store needs {_need_gb:.1f} GB)')
shutil.rmtree(STORE, ignore_errors=True)
os.makedirs(STORE)
_t = time.time()
with ThreadPoolExecutor(4) as _ex:
    _res = list(_ex.map(lambda n: extract_tar(n, TARS[n], STORE), sorted(TARS)))
_dt = time.time() - _t
_bad = [(n, h) for n, h, _, _ in _res if h != EXP['parts'][n]]
if _bad:
    abort(f'tar sha256 mismatch vs the prep manifests: {_bad}')
_names = sorted(f for f in os.listdir(STORE) if f.endswith('.npz'))
_left = [f for f in os.listdir(STORE) if f.endswith('.part')]
_nm_sha = hashlib.sha256('\n'.join(_names).encode()).hexdigest()
if _left or len(_names) != BOX['store_n'] or sum(r[2] for r in _res) != BOX['store_n'] or _nm_sha != BOX['store_names_sha']:
    abort(f'store names differ from the box: {len(_names)} npz (box {BOX["store_n"]}), members '
          f'{sum(r[2] for r in _res)}, names sha {_nm_sha[:16]} (box {BOX["store_names_sha"][:16]}), partial {_left[:3]}')
_smp_bad = [s['name'] for s in BOX['store_sample'] if sha256_file(f'{STORE}/{s["name"]}') != s['sha256']]
if _smp_bad:
    abort(f'store files differ from the box ~/fleet_store: {_smp_bad}')
import pandas as pd
_lab = pd.read_csv(f'{LBL}/labels_mix50_logit.csv', dtype={'StudyInstanceUID': str}).StudyInstanceUID
_miss = [s for s in _lab if not os.path.exists(f'{STORE}/{s}.npz')]
if len(_miss) != BOX['labelled_missing']:
    abort(f'{len(_miss)} labelled studies missing from the store (box: {BOX["labelled_missing"]}), e.g. {_miss[:3]}')
_gb = sum(r[3] for r in _res) / 1e9
STORE_INFO = dict(dir=STORE, n_npz=len(_names), names_sha=_nm_sha, extract_s=round(_dt, 1), tar_gb=round(_gb, 3),
                  mb_per_s=round(_gb * 1e3 / max(_dt, 1e-6), 1), sample_ok=len(BOX['store_sample']),
                  parts_ok=len(_res), labelled_missing=len(_miss), free_gb_before=round(_free_gb, 1),
                  free_gb_candidates={k: round(v, 1) for k, v in _free.items()})
REC['store'] = STORE_INFO
stage('store', ok=True)
log(f'store OK: {len(_names)} npz from {len(_res)} tars in {_dt:.0f} s ({STORE_INFO["mb_per_s"]} MB/s), '
    f'{len(BOX["store_sample"])}/{len(BOX["store_sample"])} sampled files == box, names sha == box')


In [ ]:
# [step 4] parity before any GPU work: (a) store/tensor contract, select_sha, jpeg_fp, model_sha == the box ckpt;
# (b) the box ckpt's decode probe (4 studies decoded in THIS cv2 build) == the box decode; (c) the model built here
# (timm + bundled pretrained bytes) has the box state_dict keys/shapes/dtypes and the pretrained weights loaded;
# (d) fold counts == the box arithmetic; (e) one-thread CPU cost of the heavy train path vs the box.
if CODE not in sys.path:
    sys.path.insert(0, CODE)
import inspect
import fleet_common as fc
import fleet_model as _fm, fleet_model_ttr as _ftt
_a = BOX['ckpt_args']
_msha = hashlib.sha256((chr(10).join(inspect.getsource(_ftt).splitlines()) + chr(10) +
                        chr(10).join(inspect.getsource(_fm).splitlines())).encode()).hexdigest()[:16]
PAR = dict(contract=json.loads(json.dumps(fc.CONTRACT)), select_sha=fc.SELECT_SHA, jpeg_fp=fc.jpeg_fingerprint(),
           model_sha=_msha)
_bad = []
if PAR['contract'] != _a['contract']:
    _bad.append(f'contract {PAR["contract"]} != box {_a["contract"]}')
for _k in ('select_sha', 'jpeg_fp', 'model_sha'):
    if PAR[_k] != _a[_k]:
        _bad.append(f'{_k} {PAR[_k]} != box {_a[_k]}')
_dp = {}
for _s, _h in _a['decode_probe'].items():
    with np.load(f'{STORE}/{_s}.npz') as _z:
        fc.check_store_contract(_z, _s)
        _dp[_s] = fc.decode_sha(_z)
    if _dp[_s] != _h:
        _bad.append(f'decode probe {_s[-12:]}: {_dp[_s]} != box {_h}')
PAR['decode_probe_match'] = sum(_dp[s] == h for s, h in _a['decode_probe'].items())
if _bad:
    REC['parity'] = dict(PAR, problems=_bad)
    abort('parity with the box failed: ' + ' | '.join(_bad))
log(f'parity OK: pixel_sha {PAR["contract"]["pixel_sha"]} tensor_sha {PAR["contract"]["tensor_sha"]} select_sha '
    f'{PAR["select_sha"]} jpeg_fp {PAR["jpeg_fp"]} model_sha {PAR["model_sha"]} | decode probe '
    f'{PAR["decode_probe_match"]}/{len(_a["decode_probe"])}')

_pre = install_hf_cache(DS, EXP)
_mc = None
for _off in ('1', '0'):
    _r = subprocess.run([sys.executable, f'{CODE}/kt_modelcheck.py', CODE, f'{DS}/box_facts.json', _pre],
                        capture_output=True, text=True, cwd=CODE,
                        env=dict(os.environ, CUDA_VISIBLE_DEVICES='', HF_HOME=HF_HOME, HF_HUB_OFFLINE=_off,
                                 OMP_NUM_THREADS='2'))
    _line = [l for l in _r.stdout.splitlines() if l.startswith('KTMODELCHECK ')]
    if _r.returncode == 0 and _line:
        _mc = json.loads(_line[-1][len('KTMODELCHECK '):])
        HF_OFFLINE[0] = _off
        break
    warn(f'model check with HF_HUB_OFFLINE={_off} failed (rc {_r.returncode}): {_r.stderr[-1500:]}')
if _mc is None:
    abort('could not build KneeFleetNetTTR with pretrained weights')
if HF_OFFLINE[0] != '1':
    warn('pretrained weights came from the HF hub download, not the bundled box bytes (equality checked below)')
PAR['model_check'] = _mc
PAR['hf_offline'] = HF_OFFLINE[0]
if not (_mc['keys_match'] and _mc['pretrained_equal']):
    REC['parity'] = PAR
    abort(f'model check failed: {_mc}')
log(f'model OK: {_mc["n_keys"]} state_dict keys/shapes/dtypes == box ckpt; {_mc["cnn_keys"]} backbone tensors == the '
    f'box pretrained file (timm {_mc["timm"]}, torch {_mc["torch"]}, offline {HF_OFFLINE[0]})')

COUNTS = {}
for F in FOLDS:
    COUNTS[F] = expected_counts(LBL, STORE, F)
    _bx = BOX['expected_counts'][str(F)]
    if COUNTS[F] != _bx:
        abort(f'fold {F} counts {COUNTS[F]} != box {_bx}')
    log(f'fold {F}: {COUNTS[F]}')
PAR['counts'] = COUNTS

import cv2 as _cv2
from fleet_aug import heavy_study
_cv2.setNumThreads(0)
_nt = torch.get_num_threads()
torch.set_num_threads(1)
_ids = sorted(f[:-4] for f in os.listdir(STORE) if f.endswith('.npz'))
_bench = _ids[200:216] if len(_ids) >= 216 else _ids[:16]
np.random.seed(0)
_t = time.time()
for _s in _bench:
    with np.load(f'{STORE}/{_s}.npz') as _z:
        fc.check_store_contract(_z, _s)
        heavy_study(_z, _s, 16, [p for p, _ in fc.VIEWS])
_th = (time.time() - _t) / len(_bench)
_t = time.time()
for _s in _bench:
    with np.load(f'{STORE}/{_s}.npz') as _z:
        for _j in range(len(fc.VIEWS)):
            if f'v{_j}_blob' in _z.files:
                fc.sample_triplets(fc.decode_stack(_z[f'v{_j}_blob'], _z[f'v{_j}_offs']), fc.N_SLICES, 0.0)
_te = (time.time() - _t) / len(_bench)
torch.set_num_threads(_nt)
_bb = BOX['cpu_bench_box']
BENCH = dict(n=len(_bench), heavy_s_per_study=round(_th, 4), eval_s_per_study=round(_te, 4),
             box_heavy_s_per_study=_bb['heavy_s_per_study'], box_eval_s_per_study=_bb['eval_s_per_study'],
             cpu_bound_s_it_floor=round(2 * _th * len(FOLDS) / max(1, os.cpu_count()), 3))
PAR['cpu_bench'] = BENCH
REC['parity'] = PAR
stage('checks', ok=True)
log(f'CPU heavy train path {_th:.3f} s/study/thread (box {_bb["heavy_s_per_study"]}), eval path {_te:.3f} '
    f'(box {_bb["eval_s_per_study"]}); with {os.cpu_count()} vCPUs and {len(FOLDS)} trainers at bs 2 the decode-only '
    f'floor is ~{BENCH["cpu_bound_s_it_floor"]} s/it per trainer')


In [ ]:
# [step 5] timing probe: BOTH trainers at once with the real recipe (fold 3 on GPU0, fold 4 on GPU1) until each has
# printed 'it PROBE_ITERS' (or timeout), then killed. Steady s/it = window between the last two progress lines.
# The probe logs also prove, before any real training, that the trainer itself prints the box contract and the
# expected fold counts (abort otherwise).
if not wait_gpus_free(60):
    abort(f'GPUs not free before the probe: {gpu_query()}')
shutil.rmtree(PROBE_DIR, ignore_errors=True)
os.makedirs(PROBE_DIR)
_W = CFG['WORKERS']
_pp = {F: launch(F, g, CFG['EPOCHS_MAX'], save_epochs_arg(CFG['EPOCHS_MAX']), f'{PROBE_DIR}/out_f{F}',
                 f'{PROBE_DIR}/f{F}.log', LBL, _W) for F, g in zip(FOLDS, CFG['GPU_IDS'])}
_t_start = time.time()
_first, _cpu_a, _cpu_b, _gutil, _t_last_g, _hdr_ok = {}, None, None, {}, 0.0, {}
_why = 'timeout'
while True:
    time.sleep(CFG['PROBE_POLL_S'])
    _txt = {F: read_text(f'{PROBE_DIR}/f{F}.log') for F in FOLDS}
    _its = {F: parse_its(_txt[F]) for F in FOLDS}
    for F in FOLDS:
        if F not in _hdr_ok and RE_IT.search(_txt[F]):          # header is complete once training prints
            _pb = header_problems(parse_header(_txt[F]), COUNTS[F], BOX)
            if _pb:
                for G in FOLDS:
                    stop_group(_pp[G], min(10, CFG['KILL_GRACE_S']))
                REC['probe'] = dict(header_problems={F: _pb})
                abort(f'probe fold {F}: trainer header differs from the box: {_pb}')
            _hdr_ok[F] = True
        if _its[F] and F not in _first:
            _first[F] = time.time()
    if _cpu_a is None and len(_first) == len(FOLDS):
        _cpu_a, _t_cpu_a = cpu_times(), time.time()
    if _cpu_a is not None and time.time() - _t_last_g >= 10:
        for g in gpu_query():
            _gutil.setdefault(g['index'], []).append(float(g['util_pct']))
        _t_last_g = time.time()
    if all(_its[F] and _its[F][-1][0] >= CFG['PROBE_ITERS'] for F in FOLDS):
        _why = 'reached'
        break
    _dead = [F for F in FOLDS if _pp[F].poll() is not None]
    if _dead:
        _why = f'trainer exited early: folds {_dead}'
        break
    if time.time() - _t_start > CFG['PROBE_TIMEOUT_S']:
        break
_cpu_b, _t_cpu_b = cpu_times(), time.time()
_how = {F: stop_group(_pp[F], min(20, CFG['KILL_GRACE_S'])) for F in FOLDS}
_txt = {F: read_text(f'{PROBE_DIR}/f{F}.log') for F in FOLDS}
if _why.startswith('trainer exited'):
    for F in FOLDS:
        log(f'--- probe fold {F} log tail ---\n' + tail(f'{PROBE_DIR}/f{F}.log', 40))
    REC['probe'] = dict(why=_why, rc={F: _pp[F].returncode for F in FOLDS})
    abort(f'probe: {_why} (see the log tails above)')
_sit, _src = {}, {}
for F in FOLDS:
    _its = parse_its(_txt[F])
    if F not in _hdr_ok:
        _pb = header_problems(parse_header(_txt[F]), COUNTS[F], BOX)
        if _pb:
            abort(f'probe fold {F}: header problems {_pb}')
    w = sit_window(_its)
    if w is not None and w > 0:
        _sit[F], _src[F] = w, f'window it {_its[-2][0]}->{_its[-1][0]}'
    elif _its and F in _first:
        _sit[F] = 1.5 * (_t_cpu_b - _first[F]) / 200.0       # never reached it 200: s/it > elapsed / 200
        _src[F] = f'only it {_its[-1][0]} after {_t_cpu_b - _first[F]:.0f} s: pessimistic 1.5 x elapsed / 200'
        warn(f'probe fold {F}: {_src[F]} -> {_sit[F]:.2f} s/it')
    else:
        abort(f'probe fold {F}: no progress line within {CFG["PROBE_TIMEOUT_S"]:.0f} s:\n' + tail(f'{PROBE_DIR}/f{F}.log'))
if not wait_gpus_free(120):
    warn(f'GPU memory still in use after the probe: {gpu_query()}')
_busy = cpu_busy(_cpu_a, _cpu_b) if _cpu_a is not None else None
_gu = {k: round(sum(v) / len(v), 1) for k, v in _gutil.items() if v}
S_IT = max(_sit.values())
_studies_s = sum(2.0 / v for v in _sit.values())
_verdict = ('CPU-bound (decode/augment)' if (_busy or 0) > 0.9 and (not _gu or max(_gu.values()) < 80) else
            'GPU-bound' if _gu and min(_gu.values()) >= 85 else 'mixed')
PROBE = dict(why=_why, workers=_W, s_per_it=_sit, source=_src, s_it_used=S_IT, stopped=_how,
             probe_wall_s=round(_t_cpu_b - _t_start, 1), cpu_busy_frac=None if _busy is None else round(_busy, 3),
             gpu_util_mean_pct=_gu, train_studies_per_s_total=round(_studies_s, 2),
             decode_s_per_study_thread=BENCH['heavy_s_per_study'],
             decode_cpu_s_per_s=round(_studies_s * BENCH['heavy_s_per_study'], 2), verdict=_verdict,
             progress_lines={F: parse_its(_txt[F]) for F in FOLDS})
REC['probe'] = PROBE
os.makedirs(f'{WORK}/probe', exist_ok=True)
for F in FOLDS:
    shutil.copyfile(f'{PROBE_DIR}/f{F}.log', f'{WORK}/probe/probe_f{F}.log')
stage('probe', ok=True)
log(f'probe ({_why}, {PROBE["probe_wall_s"]:.0f} s, workers {_W}/trainer): s/it {{{", ".join(f"f{F}: {v:.3f}" for F, v in _sit.items())}}} '
    f'| CPU busy {PROBE["cpu_busy_frac"]} | GPU util {_gu} | {_studies_s:.2f} train studies/s total = '
    f'{PROBE["decode_cpu_s_per_s"]} CPU-s/s of decode+augment at the one-thread bench | {_verdict}')


In [ ]:
# [step 6] choose E from the probe, launch fold 3 (GPU0) and fold 4 (GPU1) for real, watch them, enforce the hard
# wall, then write the receipt. Outputs go straight to /kaggle/working/f{F}_s1043_ttr_mix50/ (+ .log next to it):
# the trainer rewrites its best raw/ema ckpt + OOF csv whenever val improves, so a kill keeps best-so-far.
_el0 = el()
_iters = max(COUNTS[F]['iters_per_epoch'] for F in FOLDS)
_ec = choose_epochs(S_IT, _iters, _el0)
E = _ec['E']
SAVE_EP = save_epochs_arg(E)
PLAN = dict(E=E, save_epochs=SAVE_EP, s_it=S_IT, iters_per_epoch=_iters, eval_frac=CFG['EVAL_FRAC'],
            epoch_min_est=round(_ec['epoch_s'] / 60, 2), elapsed_at_launch_h=round(_el0 / 3600, 3),
            avail_h=round(_ec['avail_s'] / 3600, 3), budget_h=CFG['BUDGET_H'], wall_h=CFG['WALL_H'],
            est_finish_h=round(_ec['finish_elapsed_s'] / 3600, 2), est_finish_ist=ist(T0 + _ec['finish_elapsed_s']),
            wall_ist=ist(T0 + CFG['WALL_H'] * 3600), box_recipe_epochs=_a['epochs'])
REC['plan'] = PLAN
log(f'E = {E} epochs (min({CFG["EPOCHS_MAX"]}, floor({_ec["avail_s"] / 3600:.2f} h / {_ec["epoch_s"] / 60:.1f} min))), '
    f'--save_epochs {SAVE_EP}; est. finish {PLAN["est_finish_ist"]} (+{PLAN["est_finish_h"]} h), wall {PLAN["wall_ist"]}')
if E < CFG['EPOCHS_WARN']:
    warn(f'E = {E} < {CFG["EPOCHS_WARN"]}: the cosine schedule is {E}/24 of the box recipe; the operator decides '
         f'whether these fold-3/4 members are used')
save_receipt()

if not wait_gpus_free(120):
    warn(f'GPU memory in use before the real launch: {gpu_query()}')
for F in FOLDS:
    if os.path.exists(f'{WORK}/{RUN[F]}'):
        shutil.rmtree(f'{WORK}/{RUN[F]}')
PROCS = {F: launch(F, g, E, SAVE_EP, f'{WORK}/{RUN[F]}', f'{WORK}/{RUN[F]}.log', LBL, CFG['WORKERS'])
         for F, g in zip(FOLDS, CFG['GPU_IDS'])}
T_LAUNCH = time.time()
REC['train'] = dict(launch_ist=ist(T_LAUNCH), launch_h=round(el() / 3600, 3),
                    cmd={F: ' '.join(PROCS[F].kt_cmd) for F in FOLDS}, folds={})
REC['status'] = 'training'
save_receipt()
ENDED, _hdr_ok, _proj_warned, _last_status = {}, {}, False, 0.0
WALL = T0 + CFG['WALL_H'] * 3600


def _snapshot():
    out = {}
    for F in FOLDS:
        txt = read_text(f'{WORK}/{RUN[F]}.log')
        pr = progress(txt)
        its = parse_its(txt, max(0, pr['last_ep_done'] + 1))
        ep_min = [m for ep, tag, _, m in pr['evals'] if tag == 'ema']
        out[F] = dict(epochs_done=pr['last_ep_done'] + 1, best=pr['best'], epoch_min=ep_min,
                      train_only_min=pr['train_only_min'], cur_it=its[-1][0] if its else None,
                      alive=PROCS[F].poll() is None, rc=PROCS[F].returncode)
    return out


def finalize():
    import torch
    snap = _snapshot()
    total, folds = 0, {}
    exp_keys = [k for k, _, _ in BOX['state_dict']]
    for F in FOLDS:
        d, lp = f'{WORK}/{RUN[F]}', f'{WORK}/{RUN[F]}.log'
        txt = read_text(lp)
        files = []
        for root, _, fs in os.walk(d):
            for f in sorted(fs):
                p = os.path.join(root, f)
                rel = os.path.relpath(p, WORK).replace(os.sep, '/')
                e = dict(path=rel, bytes=os.path.getsize(p), sha256=sha256_file(p))
                if f.endswith('.pt'):
                    try:
                        ck = torch.load(p, map_location='cpu', weights_only=False)
                        e.update(load_ok=True, keys_match=list(ck['model'].keys()) == exp_keys, cv=ck.get('cv'),
                                 epoch=ck.get('epoch'), args_epochs=ck['args'].get('epochs'),
                                 args_fold=ck['args'].get('fold'), args_seed=ck['args'].get('seed'))
                        del ck
                    except Exception as ex:
                        e.update(load_ok=False, error=f'{type(ex).__name__}: {str(ex)[:200]}')
                        warn(f'{rel} does not load: {e["error"]}')
                files.append(e)
                total += e['bytes']
        cl = closing_line(txt)
        best_pt = [x['path'] for x in files if x['path'].endswith('_best.pt') and '/epochs/' not in x['path']]
        folds[F] = dict(run=RUN[F], out_dir=d, log=lp, rc=PROCS[F].returncode, ended=ENDED.get(F),
                        closing_line=cl, complete=cl is not None, best_ckpts=best_pt,
                        oof=[x['path'] for x in files if os.path.basename(x['path']).startswith('oof_')
                             and '/epochs/' not in x['path']],
                        progress=snap[F], files=files, log_tail=tail(lp, 6))
        if os.path.exists(lp):
            total += os.path.getsize(lp)
    if total > CFG['OUT_LIMIT_GB'] * 1e9:                 # never expected (~1.4 GB); keep best ckpts + all csv
        for F in FOLDS:
            for p in glob.glob(f'{WORK}/{RUN[F]}/epochs/*.pt'):
                os.remove(p)
        warn(f'outputs {total / 1e9:.1f} GB > {CFG["OUT_LIMIT_GB"]} GB: removed epochs/*.pt')
    REC['train']['folds'] = folds
    REC['train']['end_ist'] = ist()
    REC['train']['hours'] = round((time.time() - T_LAUNCH) / 3600, 3)
    REC['outputs_gb'] = round(total / 1e9, 3)
    if not str(REC['status']).startswith('ABORTED'):
        REC['status'] = ('DONE' if all(folds[F]['complete'] for F in FOLDS) else
                         'PARTIAL (no closing line for folds ' +
                         ', '.join(str(F) for F in FOLDS if not folds[F]['complete']) + ')')
    save_receipt()
    for F in FOLDS:
        x = folds[F]
        log(f'fold {F}: {x["ended"]} rc {x["rc"]} | closing {x["closing_line"]} | epochs done '
            f'{x["progress"]["epochs_done"]}/{E} | best {x["progress"]["best"]} | {len(x["files"])} files')
    log(f'STATUS {REC["status"]} | outputs {REC["outputs_gb"]} GB | receipt {WORK}/run_receipt.json')


try:
    while True:
        now = time.time()
        for F in FOLDS:
            if F not in _hdr_ok:
                txt = read_text(f'{WORK}/{RUN[F]}.log')
                if RE_IT.search(txt):
                    _pb = header_problems(parse_header(txt), COUNTS[F], BOX)
                    if _pb:
                        for G in FOLDS:
                            ENDED[G] = stop_group(PROCS[G], 10)
                        abort(f'fold {F}: trainer header differs from the box: {_pb}')
                    _hdr_ok[F] = True
        snap = _snapshot()
        REC['train']['progress'] = snap
        if not any(s['alive'] for s in snap.values()):
            for F in FOLDS:
                ENDED[F] = stop_group(PROCS[F], 5)
            break
        if now >= WALL:
            warn(f'hard wall {CFG["WALL_H"]} h reached: terminating the trainers (best-so-far ckpts are kept)')
            _wq = wait_quiet([f'{WORK}/{RUN[F]}' for F in FOLDS])
            log(f'outputs quiet after {_wq:.0f} s (no ckpt/OOF write in flight): sending SIGTERM')
            REC['train']['wall_quiet_wait_s'] = round(_wq, 1)
            for F in FOLDS:
                ENDED[F] = stop_group(PROCS[F], CFG['KILL_GRACE_S'])
            break
        _done = [s['epochs_done'] for s in snap.values() if s['epochs_done'] > 0]
        if _done and not _proj_warned:
            _per = max(sum(s['epoch_min']) / len(s['epoch_min']) for s in snap.values() if s['epoch_min']) * 60
            _fin = T_LAUNCH + CFG['STARTUP_S'] + E * _per
            REC['train']['projected_finish_ist'] = ist(_fin)
            if _fin > WALL:
                _proj_warned = True
                warn(f'measured {_per / 60:.1f} min/epoch -> projected finish {ist(_fin)} is past the wall '
                     f'{ist(WALL)}: the last epochs will be cut (best-so-far is kept)')
        if now - _last_status >= CFG['STATUS_EVERY_S']:
            _last_status = now
            log(' | '.join(f'f{F}: ep {s["epochs_done"]}/{E} it {s["cur_it"]} best ' +
                           ','.join(f'{t} {v[0]:.4f}@{v[1]}' for t, v in sorted(s['best'].items()))
                           for F, s in snap.items()) +
                f' | proj. finish {REC["train"].get("projected_finish_ist", "?")}')
        save_receipt()
        time.sleep(CFG['POLL_S'])
finally:
    for F in FOLDS:
        if F not in ENDED:
            ENDED[F] = stop_group(PROCS[F], CFG['KILL_GRACE_S'])
    finalize()


In [ ]:
# [step 7] summary for the commit log (the receipt has everything).
shutil.rmtree(PROBE_DIR, ignore_errors=True)
print(json.dumps(dict(status=REC['status'], plan=REC.get('plan'), probe={k: REC['probe'][k] for k in
                      ('s_per_it', 's_it_used', 'cpu_busy_frac', 'gpu_util_mean_pct', 'verdict', 'workers')},
                      warnings=REC['warnings'], outputs_gb=REC.get('outputs_gb')), indent=1, default=str))
for F in FOLDS:
    x = REC['train']['folds'][F]
    print(f'fold {F}: complete={x["complete"]} ended={x["ended"]} best ckpts {x["best_ckpts"]} oof {x["oof"]}')
for root, _, fs in os.walk(WORK):
    for f in sorted(fs):
        p = os.path.join(root, f)
        print(f'{os.path.getsize(p) / 1e6:10.1f} MB  {os.path.relpath(p, WORK)}')
